# ML1 – Hand Gesture Classification with HaGRID Landmarks (No MLflow)

This notebook trains and evaluates several classical ML models to classify hand gestures using MediaPipe hand landmarks extracted from the HaGRID dataset.

It mirrors the local Python scripts in this repository but is designed to run end-to-end in Google Colab.

**Important:** This notebook intentionally does **not** contain any MLflow code on the `main` branch. The MLflow-enabled version lives on the `research` branch and imports helper functions from `mlflow_utils.py`.


In [ ]:
# If running in Colab, install dependencies (uncomment if needed)
# !pip install numpy pandas scikit-learn matplotlib seaborn joblib mediapipe opencv-python

import pandas as pd
from pathlib import Path

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
DATA_PATH = PROJECT_ROOT / "data" / "hand_landmarks.csv"

WRIST_INDEX = 0
MIDDLE_FINGER_TIP_INDEX = 12

In [ ]:
def load_hand_landmarks(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path)


def recenter_and_normalize(df: pd.DataFrame, label_col: str = "gesture"):
    feature_cols = [c for c in df.columns if c != label_col]

    wrist_x = df[f"x_{WRIST_INDEX}"]
    wrist_y = df[f"y_{WRIST_INDEX}"]

    for i in range(21):
        df[f"x_{i}"] = df[f"x_{i}"] - wrist_x
        df[f"y_{i}"] = df[f"y_{i}"] - wrist_y

    middle_x = df[f"x_{MIDDLE_FINGER_TIP_INDEX}"]
    middle_y = df[f"y_{MIDDLE_FINGER_TIP_INDEX}"]
    scale = (middle_x ** 2 + middle_y ** 2) ** 0.5
    scale = scale.replace(0, 1e-6)

    for i in range(21):
        df[f"x_{i}"] = df[f"x_{i}"] / scale
        df[f"y_{i}"] = df[f"y_{i}"] / scale

    X = df[feature_cols]
    y = df[label_col]
    return X, y

In [ ]:
from sklearn.model_selection import train_test_split


df = load_hand_landmarks(str(DATA_PATH))
X, y = recenter_and_normalize(df)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
models = {
    "RandomForest": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    "SVM-RBF": SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=15),
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_macro": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))

    results.append(metrics)

results_df = pd.DataFrame(results)
results_df

In [ ]:
plt.figure(figsize=(8, 5))
metrics_to_plot = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]
long_df = results_df.melt(id_vars="model", value_vars=metrics_to_plot, var_name="metric")
sns.barplot(data=long_df, x="model", y="value", hue="metric")
plt.ylim(0, 1.0)
plt.title("Model comparison (macro metrics)")
plt.show()

best_row = results_df.sort_values("f1_macro", ascending=False).iloc[0]
print("Best model:")
best_row

In [ ]:
import joblib

best_model_name = best_row["model"]
print("Saving best model:", best_model_name)

from pathlib import Path

MODELS_DIR = (PROJECT_ROOT / "models")
MODELS_DIR.mkdir(exist_ok=True)

best_model = models[best_model_name]
model_path = MODELS_DIR / "best_model.joblib"
joblib.dump(best_model, model_path)

print("Model saved to", model_path)